In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [15]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("GROUBD_TRUTH_WITH_FINAL_LABEL.csv", index_col=0)
print(f"✅ Loaded {len(df)} reviews")

# Save original labels
df["original_final_label"] = df["final_label"]

# Drop the 4 label columns
columns_to_drop = ['label_1', 'label_2', 'label_3', 'final_label']
df.drop(columns=columns_to_drop, inplace=True)
print(f"Dataset shape: {df.shape}")

# Load model
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f" Model loaded successfully!")

print("STEP 3: Making predictions on 200 reviews")
print("=" * 60)

def predict_sentiment(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return "neutral"
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    return label_map[predicted_class]

predictions = []
total = len(df)

for i, row in df.iterrows():
    # Show progress every 20 reviews
    if (i + 1) % 20 == 0 or i == 0:
        print(f"⏳ Processing: {i+1}/{total} reviews...")
    
    pred = predict_sentiment(row["review_text"])
    predictions.append(pred)

df["predicted_label"] = predictions
print(f"✅ All {total} reviews processed!")

print("\n" + "=" * 60)
print("STEP 4: Sample results (first 10 reviews)")
print("=" * 60)

for i in range(min(10, len(df))):
    review_text = df.iloc[i]['review_text']
    if len(str(review_text)) > 60:
        review_text = str(review_text)[:60] + "..."
    print(f"\n{i+1}. Review: {review_text}")
    print(f"   Predicted by RoBERTa: {df.iloc[i]['predicted_label']}")
    print(f"   Original label:      {df.iloc[i]['original_final_label']}")

print("\n" + "=" * 60)
print("STEP 5: Model Evaluation (Accuracy)")
print("=" * 60)

# Calculate accuracy
accuracy = accuracy_score(df["original_final_label"], df["predicted_label"])
print(f"\n🎯 Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\n Detailed Classification Report:")
print(classification_report(df["original_final_label"], df["predicted_label"]))

print("\n Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(df["original_final_label"], df["predicted_label"]))

print("\n" + "=" * 60)
print("STEP 6: Saving results")
print("=" * 60)

# Save to CSV
df.to_csv("roberta_predictions.csv", index=False)
print("✅ Results saved to: roberta_predictions.csv")

print("\n" + "=" * 60)
print("STEP 7: Prediction Distribution")
print("=" * 60)
print(df["predicted_label"].value_counts())

print("\n" + "=" * 60)
print("✅ ALL DONE!")
print("=" * 60)

✅ Loaded 200 reviews
Dataset shape: (200, 17)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Model loaded successfully!
STEP 3: Making predictions on 200 reviews
⏳ Processing: 1/200 reviews...
⏳ Processing: 20/200 reviews...
⏳ Processing: 40/200 reviews...
⏳ Processing: 60/200 reviews...
⏳ Processing: 80/200 reviews...
⏳ Processing: 100/200 reviews...
⏳ Processing: 120/200 reviews...
⏳ Processing: 140/200 reviews...
⏳ Processing: 160/200 reviews...
⏳ Processing: 180/200 reviews...
⏳ Processing: 200/200 reviews...
✅ All 200 reviews processed!

STEP 4: Sample results (first 10 reviews)

1. Review: `
   Predicted by RoBERTa: neutral
   Original label:      neutral

2. Review: incredibly fun and great replayability (i think that's how y...
   Predicted by RoBERTa: positive
   Original label:      positive

3. Review: I Monster my Hunter till I Wilds
   Predicted by RoBERTa: neutral
   Original label:      neutral

4. Review: This is a really good game if you love Zombie Apocolypse gam...
   Predicted by RoBERTa: positive
   Original label:      positive

5. Review: love the game 